# 12_Doo — 튜닝 모델 결과 비교

## 목적
`08~11` 튜닝 노트북의 Validation 결과를 같은 기준으로 비교해 최종 모델 후보를 선정한다. Test 데이터는 아직 사용하지 않는다.

## 평가 기준
고객 이탈을 놓치지 않기 위해 Recall을 우선하고, Precision·F1·ROC-AUC와 Train-Validation gap을 함께 확인한다.

## 1. 결과 출처와 입력 기준
아래 표는 우리가 만든 `08~11` 튜닝 노트북의 실행 결과를 정리한 것이다. `source_notebook`은 원본 파일, `best_parameters`는 해당 파일의 Best parameters, 나머지 항목은 Validation 성능과 선택 Threshold다.
성능 지표는 임계값 조정 결과이며, 하이퍼파라미터와 혼동하지 않도록 별도 컬럼으로 구분한다.

## 2. 최적 하이퍼파라미터 요약

| 모델 | 최적 파라미터 |
|---|---|
| XGBoost | `n_estimators=250`, `learning_rate=0.02`, `max_depth=4`, `min_child_weight=3`, `subsample=0.70`, `colsample_bytree=0.90`, `reg_alpha=0.10`, `reg_lambda=3` |
| Logistic Regression | `C=0.007279`, L2 정규화, `solver=liblinear` |
| Random Forest | `n_estimators=800`, `max_depth=5`, `min_samples_split=10`, `min_samples_leaf=8`, `max_features=None`, `class_weight=balanced_subsample` |
| SVC | `C=37.9269`, `gamma=0.021544`, `class_weight=balanced`, `kernel=rbf` |

위 값은 각각 `08~11`번 튜닝 노트북의 Train 내부 5-fold 교차검증에서 선택된 Best parameters다.

In [1]:
import pandas as pd

# 각 08~11 노트북의 튜닝 결과를 입력한다.
# 아래 값은 현재 실행 결과를 반영한 예시이며, 팀원의 최신 실행 결과로 갱신한다.
tuned_results = pd.DataFrame([
    {'model': 'XGBoost tuned', 'source_notebook': '08_Doo_xgboost_tuning.ipynb',
     'best_parameters': 'n_estimators=250, learning_rate=0.02, max_depth=4, min_child_weight=3, subsample=0.70, colsample_bytree=0.90, reg_alpha=0.10, reg_lambda=3',
     'val_accuracy': 0.704, 'recall': 0.874, 'precision': 0.633, 'f1': 0.734, 'roc_auc': 0.777,
     'selected_threshold': 0.35, 'train_val_gap': 0.059},
    {'model': 'Logistic Regression tuned', 'source_notebook': '09_Doo_logistic_tuning.ipynb',
     'best_parameters': 'C=0.007279, L2 regularization, solver=liblinear', 'val_accuracy': 0.703, 'recall': 0.836,
     'precision': 0.636, 'f1': 0.723, 'roc_auc': 0.761, 'selected_threshold': 0.40, 'train_val_gap': -0.001},
    {'model': 'Random Forest tuned', 'source_notebook': '10_Doo_random_forest_tuning.ipynb',
     'best_parameters': 'n_estimators=800, max_depth=5, min_samples_split=10, min_samples_leaf=8, max_features=None, class_weight=balanced_subsample',
     'val_accuracy': 0.706, 'recall': 0.834, 'precision': 0.650, 'f1': 0.730, 'roc_auc': 0.769,
     'selected_threshold': 0.40, 'train_val_gap': 0.037},
    {'model': 'SVC tuned', 'source_notebook': '11_Doo_svc_tuning.ipynb',
     'best_parameters': 'C=37.9269, gamma=0.021544, class_weight=balanced, kernel=rbf', 'val_accuracy': 0.707,
     'recall': 0.817, 'precision': 0.652, 'f1': 0.726, 'roc_auc': 0.764, 'selected_threshold': 0.35,
     'train_val_gap': 0.008},
])
display(tuned_results)


,model,source_notebook,best_parameters,val_accuracy,recall,precision,f1,roc_auc,selected_threshold,train_val_gap
0,XGBoost tuned,08_Doo_xgboost_tuning.ipynb,"n_estimators=250, learning_rate=0.02, max_dept...",0.704,0.874,0.633,0.734,0.777,0.35,0.059
1,Logistic Regression tuned,09_Doo_logistic_tuning.ipynb,"C=0.007279, L2 regularization, solver=liblinear",0.703,0.836,0.636,0.723,0.761,0.40,-0.001
2,Random Forest tuned,10_Doo_random_forest_tuning.ipynb,"n_estimators=800, max_depth=5, min_samples_spl...",0.706,0.834,0.650,0.730,0.769,0.40,0.037
3,SVC tuned,11_Doo_svc_tuning.ipynb,"C=37.9269, gamma=0.021544, class_weight=balanc...",0.707,0.817,0.652,0.726,0.764,0.35,0.008


## 2. 모델 순위
Recall 0.80 이상 모델 중 F1-score가 높은 순으로 정렬한다. 실제 최종 선정에서는 팀원 모델 결과와 교차검증 안정성도 함께 확인한다.

In [2]:
ranking = tuned_results.copy()
ranking['recall_target_met'] = ranking['recall'].ge(0.80)
ranking = ranking.sort_values(['recall_target_met','f1','roc_auc'], ascending=[False,False,False], na_position='last')
display(ranking.round(3))
print('현재 입력값 기준 1순위:', ranking.iloc[0]['model'])


,model,source_notebook,best_parameters,val_accuracy,recall,precision,f1,roc_auc,selected_threshold,train_val_gap,recall_target_met
0,XGBoost tuned,08_Doo_xgboost_tuning.ipynb,"n_estimators=250, learning_rate=0.02, max_dept...",0.704,0.874,0.633,0.734,0.777,0.35,0.059,True
2,Random Forest tuned,10_Doo_random_forest_tuning.ipynb,"n_estimators=800, max_depth=5, min_samples_spl...",0.706,0.834,0.650,0.730,0.769,0.40,0.037,True
3,SVC tuned,11_Doo_svc_tuning.ipynb,"C=37.9269, gamma=0.021544, class_weight=balanc...",0.707,0.817,0.652,0.726,0.764,0.35,0.008,True
1,Logistic Regression tuned,09_Doo_logistic_tuning.ipynb,"C=0.007279, L2 regularization, solver=liblinear",0.703,0.836,0.636,0.723,0.761,0.40,-0.001,True


현재 입력값 기준 1순위: XGBoost tuned


## 3. 최종 선정 체크리스트

- Validation Recall이 프로젝트 목표를 충족하는가?
- F1-score와 Precision이 지나치게 낮지 않은가?
- Train-Validation gap이 커서 과적합된 모델은 아닌가?
- 07번 교차검증 결과에서 fold별 성능이 안정적인가?
- 팀원 모델과 동일한 데이터 버전을 사용했는가?

최종 모델을 정한 뒤에만 Train+Validation 재학습과 Test 1회 평가를 진행한다.